In [20]:
# imports and setup

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
import re
import html
import unicodedata
import warnings
import time

from scipy.stats import spearmanr

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    average_precision_score,
    roc_auc_score
)
from sklearn.model_selection import KFold, cross_validate, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.neural_network import MLPRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import (
    make_scorer,
    median_absolute_error,
    explained_variance_score,
)
from scipy.stats import spearmanr, pearsonr
from xgboost import XGBRegressor


warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)


In [2]:
# find the raw json files

# path I use locally
json_dir = Path("../../raw_new")

# fallback when running in this notebook environment
if not json_dir.exists():
    json_dir = Path("/mnt/data")

# keep comment batches only; skip the tracker metadata file
json_files = sorted([
    p for p in json_dir.glob("*.json")
    if "scraped_videos_tracker" not in p.name
])

print("JSON directory:", json_dir.resolve())
print("Number of JSON files found:", len(json_files))
print([p.name for p in json_files[:5]])


JSON directory: /Users/hargun/Desktop/data-science/EarlySlangDetection/raw_new
Number of JSON files found: 75
['1STUD_batch_20260615_220936.json', '1STUD_batch_20260616_120929.json', '1STUD_batch_20260616_125913.json', '1STUD_batch_20260629_221901.json', 'AsmonTV_batch_20260608_205250.json']


In [3]:
# load the raw comment files

dfs = []

for json_file in json_files:
    temp = pd.read_json(json_file)

    # Example filename:
    #   MrBeastGaming_batch_20260608_205250.json
    # We extract:
    #   MrBeastGaming
    temp["channel_name"] = json_file.stem.split("_batch_")[0].replace("(1)", "")

    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)

# basic cleanup
df["published_at"] = pd.to_datetime(df["published_at"], utc=True, errors="coerce")
df["likes"] = pd.to_numeric(df["likes"], errors="coerce").fillna(0).clip(lower=0).astype(int)
df["text"] = df["text"].fillna("").astype(str)

# no timestamp means no week assignment
df = df.dropna(subset=["published_at"]).reset_index(drop=True)

# simple comment id after cleaning
df["comment_id"] = df.index

display(df.head())
display(df.info())

print("Raw comments:", len(df))
print("Unique videos:", df["video_id"].nunique())
print("Unique channels:", df["channel_name"].nunique())


,video_id,text,likes,published_at,channel_name,comment_id
0,BSv6hna2Ycs,3:22 zenin clan treatment,49,2026-06-15 11:44:27+00:00,1STUD,0
1,BSv6hna2Ycs,your goat stud i wacht you 4 years,29,2026-06-15 11:03:58+00:00,1STUD,1
2,BSv6hna2Ycs,I WATCHED YA FOR 2 YEARS!!!,6,2026-06-15 12:03:19+00:00,1STUD,2
3,BSv6hna2Ycs,2:21 Bro Noob Army Being Like God Slayer😂,3,2026-06-15 14:31:33+00:00,1STUD,3
4,BSv6hna2Ycs,Yo stud even though you probably not going to ...,3,2026-06-15 14:24:59+00:00,1STUD,4


<class 'pandas.DataFrame'>
RangeIndex: 1739328 entries, 0 to 1739327
Data columns (total 6 columns):
 #   Column        Dtype              
---  ------        -----              
 0   video_id      str                
 1   text          str                
 2   likes         int64              
 3   published_at  datetime64[us, UTC]
 4   channel_name  str                
 5   comment_id    int64              
dtypes: datetime64[us, UTC](1), int64(2), str(3)
memory usage: 79.6 MB


None

Raw comments: 1739328
Unique videos: 19517
Unique channels: 32


In [4]:
# assign each comment to a week

latest = df["published_at"].max()

# week_back = 0 means most recent week.
# week_back = 1 means one week before the most recent week.
df["week_back"] = ((latest - df["published_at"]).dt.days // 7).astype(int)

# keep the most recent 20 weeks
df = df[(df["week_back"] >= 0) & (df["week_back"] < 20)].copy().reset_index(drop=True)

# reset ids after filtering
df["comment_id"] = df.index

# make time_id run from old -> new
df["time_id"] = df["week_back"].max() - df["week_back"]

display(df[["comment_id", "published_at", "week_back", "time_id", "text", "likes"]].head())

print("Comments in 20-week window:", len(df))
print("time_id range:", df["time_id"].min(), "to", df["time_id"].max())


,comment_id,published_at,week_back,time_id,text,likes
0,0,2026-06-15 11:44:27+00:00,2,17,3:22 zenin clan treatment,49
1,1,2026-06-15 11:03:58+00:00,2,17,your goat stud i wacht you 4 years,29
2,2,2026-06-15 12:03:19+00:00,2,17,I WATCHED YA FOR 2 YEARS!!!,6
3,3,2026-06-15 14:31:33+00:00,2,17,2:21 Bro Noob Army Being Like God Slayer😂,3
4,4,2026-06-15 14:24:59+00:00,2,17,Yo stud even though you probably not going to ...,3


Comments in 20-week window: 373517
time_id range: 0 to 19


In [5]:
# tokenize comments

TOKEN_RE = re.compile(
    r"""
    (?<![a-z0-9])
    \#?
    (?:
        [a-z]+(?:['’‘ʼ`´＇-][a-z]+)*[a-z0-9]*
        |
        [a-z]*\d+[a-z0-9]*
    )
    (?![a-z0-9])
    """,
    re.I | re.VERBOSE
)

URL_RE = re.compile(r"https?://\S+|www\.\S+")
TIME_RE = re.compile(r"\b\d{1,2}:\d{2}(?::\d{2})?\b")

APOSTROPHE_TRANSLATION = str.maketrans({
    "’": "'",
    "‘": "'",
    "ʼ": "'",
    "`": "'",
    "´": "'",
    "＇": "'"
})

def tokenize(text):
    """Basic tokenizer for comment text."""
    text = html.unescape(str(text))
    text = unicodedata.normalize("NFKC", text)
    text = text.translate(APOSTROPHE_TRANSLATION)
    text = text.lower()
    text = URL_RE.sub(" ", text)
    text = TIME_RE.sub(" ", text)

    return [m.group(0).lstrip("#") for m in TOKEN_RE.finditer(text)]

df["tokens"] = df["text"].apply(tokenize)

display(df[["text", "tokens"]].sample(10, random_state=42))


,text,tokens
115493,Day 7 of asking for a fractal block world theory,"[day, 7, of, asking, for, a, fractal, block, w..."
247243,Interesting,[interesting]
179793,“as of 2024” okay. so wheres the updated chart?,"[as, of, 2024, okay, so, wheres, the, updated,..."
329339,"Ryan Cohen, trust the process","[ryan, cohen, trust, the, process]"
370479,Hissan wouldve captured this guy easy once the...,"[hissan, wouldve, captured, this, guy, easy, o..."
175703,Imagine jumping as a casual Morse code for bei...,"[imagine, jumping, as, a, casual, morse, code,..."
256504,The whole game in one trailer!!! Nice,"[the, whole, game, in, one, trailer, nice]"
157569,28:37 TIMMEH????,[timmeh]
362904,We’ve seen this already bro ♻️,"[we've, seen, this, already, bro]"
194213,Continue Playing Red Dead Redemption 2 we all ...,"[continue, playing, red, dead, redemption, 2, ..."


In [6]:
# one word per comment

df_words = df.explode("tokens").rename(columns={"tokens": "word"})
df_words = df_words.dropna(subset=["word"])
df_words = df_words[df_words["word"].str.len() > 0].copy()

# count a word only once per comment
df_unique = df_words.drop_duplicates(["comment_id", "word", "time_id"])

display(df_words[["comment_id", "word", "likes", "time_id", "video_id"]].head())
display(df_unique[["comment_id", "word", "likes", "time_id", "video_id"]].head())

print("Word occurrences before de-duplication:", len(df_words))
print("Unique comment-word-week rows after de-duplication:", len(df_unique))


,comment_id,word,likes,time_id,video_id
0,0,zenin,49,17,BSv6hna2Ycs
0,0,clan,49,17,BSv6hna2Ycs
0,0,treatment,49,17,BSv6hna2Ycs
1,1,your,29,17,BSv6hna2Ycs
1,1,goat,29,17,BSv6hna2Ycs


,comment_id,word,likes,time_id,video_id
0,0,zenin,49,17,BSv6hna2Ycs
0,0,clan,49,17,BSv6hna2Ycs
0,0,treatment,49,17,BSv6hna2Ycs
1,1,your,29,17,BSv6hna2Ycs
1,1,goat,29,17,BSv6hna2Ycs


Word occurrences before de-duplication: 4833752
Unique comment-word-week rows after de-duplication: 4316439


In [7]:
# aggregate word usage by week

# weekly denominators for relative features
weekly_comments = df.groupby("time_id").agg(
    comments=("comment_id", "nunique"),
    total_likes=("likes", "sum"),
    avg_comment_likes=("likes", "mean")
).reset_index()

# word-by-week usage table
weekly_word = df_unique.groupby(["word", "time_id"]).agg(
    count=("comment_id", "nunique"),
    likes=("likes", "sum"),
    avg_likes=("likes", "mean"),
    n_videos=("video_id", "nunique")
).reset_index()

weekly_word = weekly_word.merge(weekly_comments, on="time_id", how="left")

# fraction of that week's comments containing the word
weekly_word["rel_freq"] = weekly_word["count"] / weekly_word["comments"]

# Relative likes:
#   among all comment likes in a week, what fraction came from comments containing this word?
weekly_word["rel_likes"] = weekly_word["likes"] / weekly_word["total_likes"]

display(weekly_comments.head())
display(weekly_word.head())


,time_id,comments,total_likes,avg_comment_likes
0,0,16519,4514133,273.269145
1,1,17804,5365981,301.391878
2,2,18011,4533338,251.698295
3,3,20017,5308780,265.213568
4,4,18547,4246212,228.943333


,word,time_id,count,likes,avg_likes,n_videos,comments,total_likes,avg_comment_likes,rel_freq,rel_likes
0,0,0,44,11738,266.772727,37,16519,4514133,273.269145,0.002664,0.002600
1,0,1,53,30410,573.773585,39,17804,5365981,301.391878,0.002977,0.005667
2,0,2,48,25048,521.833333,39,18011,4533338,251.698295,0.002665,0.005525
3,0,3,55,7506,136.472727,49,20017,5308780,265.213568,0.002748,0.001414
4,0,4,50,6785,135.700000,39,18547,4246212,228.943333,0.002696,0.001598


In [18]:
# quick wide table for counts and likes

count_pivot = weekly_word.pivot(
    index="word",
    columns="time_id",
    values="count"
).fillna(0).astype(int)

count_pivot.columns = [f"time_{c}_count" for c in count_pivot.columns]

likes_pivot = weekly_word.pivot(
    index="word",
    columns="time_id",
    values="likes"
).fillna(0).astype(int)

likes_pivot.columns = [f"time_{c}_likes" for c in likes_pivot.columns]

final_df = pd.concat([count_pivot, likes_pivot], axis=1).fillna(0).astype(int)
final_df = final_df.reset_index()

display(final_df.head(5))
print("final_df shape:", final_df.shape)


,word,time_0_count,time_1_count,time_2_count,time_3_count,time_4_count,time_5_count,time_6_count,time_7_count,time_8_count,time_9_count,time_10_count,time_11_count,time_12_count,time_13_count,time_14_count,time_15_count,time_16_count,time_17_count,time_18_count,time_19_count,time_0_likes,time_1_likes,time_2_likes,time_3_likes,time_4_likes,time_5_likes,time_6_likes,time_7_likes,time_8_likes,time_9_likes,time_10_likes,time_11_likes,time_12_likes,time_13_likes,time_14_likes,time_15_likes,time_16_likes,time_17_likes,time_18_likes,time_19_likes
0,00,3,5,0,0,2,1,3,2,3,2,1,6,3,4,3,1,6,1,1,2,11,305,0,0,1,5,3,275,18,86,0,7,341,751,230,419,67,1,0,18
1,000,24,20,31,22,21,40,23,20,11,20,29,41,29,33,31,50,34,39,2,0,6217,3155,25593,25740,16059,38149,9292,3130,4399,5071,32536,11151,4131,26827,5167,131425,21817,8078,1,0
2,007,8,0,2,1,0,0,0,0,13,13,3,1,2,2,3,30,17,7,0,1,1463,0,3,3,0,0,0,0,690,5604,6,19837,22,0,7,1880,2629,6,0,0
3,03,1,0,1,1,6,3,7,0,0,1,0,0,1,0,0,0,1,0,0,0,2791,0,20,0,1765,7,98,0,0,77,0,0,0,0,0,0,38,0,0,0
4,10,114,152,166,148,114,150,152,152,98,176,140,163,168,243,128,173,185,169,29,32,44757,77354,143286,56914,47758,28330,107500,55275,36534,142484,44218,28892,98852,135754,29561,73414,44587,34000,5826,1822


final_df shape: (11156, 41)


In [9]:
# remove common/non-informative words

stopwords = set("""
a about above after again against all am an and any are as at be because been before being below between both but by
can cannot could did do does doing down during each few for from further had has have having he her here hers herself
him himself his how i if in into is it its itself just me more most my myself no nor not of off on once only or other
our ours ourselves out over own same she should so some such than that the their theirs them themselves then there
these they this those through to too under until up very was we were what when where which while who whom why will
with you your yours yourself yourselves
""".split())

# Contractions and common variants.
stopwords |= {
    "i'm", "it's", "that's", "don't", "you're", "he's", "she's", "we're", "they're",
    "i've", "you've", "we've", "they've", "i'll", "you'll", "we'll", "they'll",
    "isn't", "aren't", "wasn't", "weren't", "can't", "couldn't", "wouldn't",
    "shouldn't", "won't", "didn't", "doesn't", "haven't", "hasn't", "hadn't",
    "there's", "what's", "who's", "where's", "when's", "why's", "how's",
    "im", "ive", "dont", "cant", "wont", "youre", "thats", "theres", "whats",
    "hes", "shes", "were", "theyre", "ll", "re", "ve", "ur", "amp"
}

# Extra ordinary/common words that appeared in earlier top predictions.
# This is project-specific cleanup.
extra_common_words = {
    "would", "one", "get", "even", "video", "go", "know", "never", "got", "good",
    "make", "made", "see", "think", "really", "still", "also", "much", "going",
    "people", "bro", "man", "guys", "time", "way", "back", "first", "last",
    "like", "game", "person", "now", "years", "guy", "kid", "thing", "stuff",
    "day", "watch", "look", "looks", "say", "said", "right", "actually"
}

stopwords |= extra_common_words

before_words = weekly_word["word"].nunique()

weekly_word = weekly_word[
    weekly_word["word"].map(
        lambda w: (w not in stopwords) and (len(w) >= 2 or w in {"w", "l"})
    )
].copy()

# Rare-word filter:
# Keep a word if it appears in at least 5 comments in at least one week.
max_by_word = weekly_word.groupby("word").agg(max_count=("count", "max")).reset_index()
keep_words = max_by_word.loc[max_by_word["max_count"] >= 5, "word"]

weekly_word = weekly_word[weekly_word["word"].isin(keep_words)].copy()

print("Unique words before filtering:", before_words)
print("Unique words after filtering:", weekly_word["word"].nunique())


Unique words before filtering: 104102
Unique words after filtering: 11156


In [10]:
# complete word-week panel

weeks = sorted(df["time_id"].unique())

panel_index = pd.MultiIndex.from_product(
    [sorted(keep_words), weeks],
    names=["word", "time_id"]
)

panel = weekly_word.set_index(["word", "time_id"])[[
    "count",
    "likes",
    "avg_likes",
    "n_videos",
    "rel_freq",
    "rel_likes"
]].reindex(panel_index).reset_index()

# add week totals back in
panel = panel.merge(weekly_comments, on="time_id", how="left")

# missing word-week rows mean zero usage
fill_cols = ["count", "likes", "avg_likes", "n_videos", "rel_freq", "rel_likes"]
panel[fill_cols] = panel[fill_cols].fillna(0)

panel = panel.sort_values(["word", "time_id"]).reset_index(drop=True)

display(panel.head(20))
print("Panel shape:", panel.shape)


,word,time_id,count,likes,avg_likes,n_videos,rel_freq,rel_likes,comments,total_likes,avg_comment_likes
0,00,0,3.0,11.0,3.666667,3.0,0.000182,2.436791e-06,16519,4514133,273.269145
1,00,1,5.0,305.0,61.000000,5.0,0.000281,5.683956e-05,17804,5365981,301.391878
2,00,2,0.0,0.0,0.000000,0.0,0.000000,0.000000e+00,18011,4533338,251.698295
3,00,3,0.0,0.0,0.000000,0.0,0.000000,0.000000e+00,20017,5308780,265.213568
4,00,4,2.0,1.0,0.500000,2.0,0.000108,2.355040e-07,18547,4246212,228.943333
5,00,5,1.0,5.0,5.000000,1.0,0.000050,8.456538e-07,19842,5912585,297.983318
6,00,6,3.0,3.0,1.000000,3.0,0.000143,4.109866e-07,21025,7299508,347.182307
7,00,7,2.0,275.0,137.500000,2.0,0.000104,5.912774e-05,19312,4650947,240.831970
8,00,8,3.0,18.0,6.000000,3.0,0.000175,3.288685e-06,17147,5473313,319.199452
9,00,9,2.0,86.0,43.000000,1.0,0.000106,1.780010e-05,18804,4831434,256.936503


Panel shape: (223120, 11)


In [11]:
# rolling and growth features

g = panel.groupby("word", group_keys=False)

base_cols = ["count", "rel_freq", "likes", "rel_likes", "avg_likes", "n_videos"]

for col in base_cols:
    # previous week
    panel[f"prev_{col}"] = g[col].shift(1).fillna(0)

    # Raw one-week growth:
    #   current value - previous week's value
    panel[f"{col}_growth_1w"] = panel[col] - panel[f"prev_{col}"]

    # Percentage growth:
    #   growth / previous value
    # We add 1 or 1e-6 to avoid dividing by zero.
    denom = panel[f"prev_{col}"] + (1e-6 if col in ["rel_freq", "rel_likes"] else 1)
    panel[f"{col}_pct_growth_1w"] = panel[f"{col}_growth_1w"] / denom

    # Rolling 3-week baseline:
    #   average of current week and previous two weeks.
    # This is the word's recent normal level.
    panel[f"{col}_rolling_3w"] = (
        g[col]
        .rolling(3, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

    # Log transform for skewed nonnegative features.
    # log1p(x) = log(1 + x), so zero stays zero.
    panel[f"log1p_{col}"] = np.log1p(panel[col])

# Acceleration:
#   this week's growth - previous week's growth.
# This measures whether growth itself is speeding up.
for col in ["count", "rel_freq", "likes"]:
    panel[f"{col}_accel_1w"] = (
        panel.groupby("word")[f"{col}_growth_1w"]
        .diff()
        .fillna(0)
    )

# New spike-vs-baseline features.
# These directly measure whether the current week is high relative to the recent baseline.
panel["rel_freq_vs_rolling"] = panel["rel_freq"] / (panel["rel_freq_rolling_3w"] + 1e-6)
panel["likes_vs_rolling"] = panel["likes"] / (panel["likes_rolling_3w"] + 1)
panel["n_videos_vs_rolling"] = panel["n_videos"] / (panel["n_videos_rolling_3w"] + 1)

# New excess-over-baseline features.
# These measure the absolute amount by which current week exceeds recent baseline.
panel["rel_freq_excess_over_rolling"] = (
    panel["rel_freq"] - panel["rel_freq_rolling_3w"]
).clip(lower = 0)

panel["likes_excess_over_rolling"] = (
    panel["likes"] - panel["likes_rolling_3w"]
).clip(lower = 0)

panel["n_videos_excess_over_rolling"] = (
    panel["n_videos"] - panel["n_videos_rolling_3w"]
).clip(lower = 0)

display(panel.head())


,word,time_id,count,likes,avg_likes,n_videos,rel_freq,rel_likes,comments,total_likes,avg_comment_likes,prev_count,count_growth_1w,count_pct_growth_1w,count_rolling_3w,log1p_count,prev_rel_freq,rel_freq_growth_1w,rel_freq_pct_growth_1w,rel_freq_rolling_3w,log1p_rel_freq,prev_likes,likes_growth_1w,likes_pct_growth_1w,likes_rolling_3w,log1p_likes,prev_rel_likes,rel_likes_growth_1w,rel_likes_pct_growth_1w,rel_likes_rolling_3w,log1p_rel_likes,prev_avg_likes,avg_likes_growth_1w,avg_likes_pct_growth_1w,avg_likes_rolling_3w,log1p_avg_likes,prev_n_videos,n_videos_growth_1w,n_videos_pct_growth_1w,n_videos_rolling_3w,log1p_n_videos,count_accel_1w,rel_freq_accel_1w,likes_accel_1w,rel_freq_vs_rolling,likes_vs_rolling,n_videos_vs_rolling,rel_freq_excess_over_rolling,likes_excess_over_rolling,n_videos_excess_over_rolling
0,00,0,3.0,11.0,3.666667,3.0,0.000182,2.436791e-06,16519,4514133,273.269145,0.0,3.0,3.000000,3.000000,1.386294,0.000000,0.000182,181.609056,0.000182,0.000182,0.0,11.0,11.000000,11.000000,2.484907,0.000000,2.436791e-06,2.436791,2.436791e-06,2.436788e-06,0.000000,3.666667,3.666667,3.666667,1.540445,0.0,3.0,3.000000,3.000000,1.386294,0.0,0.000000,0.0,0.994524,0.916667,0.75,0.000000,0.000000,0.000000
1,00,1,5.0,305.0,61.000000,5.0,0.000281,5.683956e-05,17804,5365981,301.391878,3.0,2.0,0.500000,4.000000,1.791759,0.000182,0.000099,0.543383,0.000231,0.000281,11.0,294.0,24.500000,158.000000,5.723585,0.000002,5.440277e-05,15.829524,2.963818e-05,5.683794e-05,3.666667,57.333333,12.285714,32.333333,4.127134,3.0,2.0,0.500000,4.000000,1.791759,-1.0,-0.000082,283.0,1.209340,1.918239,1.00,0.000050,147.000000,1.000000
2,00,2,0.0,0.0,0.000000,0.0,0.000000,0.000000e+00,18011,4533338,251.698295,5.0,-5.0,-0.833333,2.666667,0.000000,0.000281,-0.000281,-0.996452,0.000154,0.000000,305.0,-305.0,-0.996732,105.333333,0.000000,0.000057,-5.683956e-05,-0.982711,1.975878e-05,0.000000e+00,61.000000,-61.000000,-0.983871,21.555556,0.000000,5.0,-5.0,-0.833333,2.666667,0.000000,-7.0,-0.000380,-599.0,0.000000,0.000000,0.00,0.000000,0.000000,0.000000
3,00,3,0.0,0.0,0.000000,0.0,0.000000,0.000000e+00,20017,5308780,265.213568,0.0,0.0,0.000000,1.666667,0.000000,0.000000,0.000000,0.000000,0.000094,0.000000,0.0,0.0,0.000000,101.666667,0.000000,0.000000,0.000000e+00,0.000000,1.894652e-05,0.000000e+00,0.000000,0.000000,0.000000,20.333333,0.000000,0.0,0.0,0.000000,1.666667,0.000000,5.0,0.000281,305.0,0.000000,0.000000,0.00,0.000000,0.000000,0.000000
4,00,4,2.0,1.0,0.500000,2.0,0.000108,2.355040e-07,18547,4246212,228.943333,0.0,2.0,2.000000,0.666667,1.098612,0.000000,0.000108,107.834151,0.000036,0.000108,0.0,1.0,1.000000,0.333333,0.693147,0.000000,2.355040e-07,0.235504,7.850134e-08,2.355040e-07,0.000000,0.500000,0.500000,0.166667,0.405465,0.0,2.0,2.000000,0.666667,1.098612,2.0,0.000108,1.0,2.918798,0.750000,1.20,0.000072,0.666667,1.333333


In [12]:
# features used in the word-week version

feature_cols = [
    # Current usage
    "log1p_count",
    "rel_freq",

    # Current engagement
    "log1p_likes",
    "rel_likes",
    "log1p_avg_likes",

    # Current spread across videos
    "log1p_n_videos",

    # One-week raw growth
    "count_growth_1w",
    "rel_freq_growth_1w",
    "likes_growth_1w",

    # One-week percentage growth
    "count_pct_growth_1w",
    "likes_pct_growth_1w",

    # Rolling 3-week baseline features
    "count_rolling_3w",
    "rel_freq_rolling_3w",
    "likes_rolling_3w",
    "rel_likes_rolling_3w",
    "avg_likes_rolling_3w",
    "n_videos_rolling_3w",

    # Acceleration
    "count_accel_1w",
    "likes_accel_1w",

    # New spike-vs-baseline features
    "rel_freq_vs_rolling",
    "likes_vs_rolling",
    "n_videos_vs_rolling",

    # New excess-over-baseline features
    "rel_freq_excess_over_rolling",
    "likes_excess_over_rolling",
    "n_videos_excess_over_rolling",
]

feature_cols


['log1p_count',
 'rel_freq',
 'log1p_likes',
 'rel_likes',
 'log1p_avg_likes',
 'log1p_n_videos',
 'count_growth_1w',
 'rel_freq_growth_1w',
 'likes_growth_1w',
 'count_pct_growth_1w',
 'likes_pct_growth_1w',
 'count_rolling_3w',
 'rel_freq_rolling_3w',
 'likes_rolling_3w',
 'rel_likes_rolling_3w',
 'avg_likes_rolling_3w',
 'n_videos_rolling_3w',
 'count_accel_1w',
 'likes_accel_1w',
 'rel_freq_vs_rolling',
 'likes_vs_rolling',
 'n_videos_vs_rolling',
 'rel_freq_excess_over_rolling',
 'likes_excess_over_rolling',
 'n_videos_excess_over_rolling']

In [13]:
# one-row-per-word regression data

df = panel.copy()

# pivot so each word has one row
wide = df.pivot(
    index="word",
    columns="time_id",
    values=["count", "likes", "avg_likes", "n_videos", "rel_freq", "rel_likes"]
)

wide.columns = [f"week_{t}_{var}" for var, t in wide.columns]
wide = wide.fillna(0)

eps_count = 1.0
eps_rel = 1e-10


# target: future growth, using only weeks 17-19
# features below only use weeks 0-16


past_rel_cols = [f"week_{i}_rel_freq" for i in range(17)]
future_rel_cols = [f"week_{i}_rel_freq" for i in range(17, 20)]

past_avg = wide[past_rel_cols].mean(axis=1)
future_avg = wide[future_rel_cols].mean(axis=1)

y = np.log((future_avg + eps_rel) / (past_avg + eps_rel))

# build X from weeks 0-16 only


features = {}

base_vars = ["count", "likes", "avg_likes", "n_videos", "rel_freq", "rel_likes"]

# raw weekly history
for var in base_vars:
    for i in range(17):
        features[f"week_{i}_{var}"] = wide[f"week_{i}_{var}"]

# adjacent log-growth features
for var in ["count", "likes", "avg_likes", "n_videos"]:
    for i in range(1, 17):
        features[f"{var}_log_growth_{i}_{i-1}"] = np.log(
            (wide[f"week_{i}_{var}"] + eps_count)
            / (wide[f"week_{i-1}_{var}"] + eps_count)
        )

for var in ["rel_freq", "rel_likes"]:
    for i in range(1, 17):
        features[f"{var}_log_growth_{i}_{i-1}"] = np.log(
            (wide[f"week_{i}_{var}"] + eps_rel)
            / (wide[f"week_{i-1}_{var}"] + eps_rel)
        )

# likes per comment
for i in range(17):
    features[f"likes_per_comment_{i}"] = (
        wide[f"week_{i}_likes"] / (wide[f"week_{i}_count"] + eps_count)
    )

# summary features
count_cols = [f"week_{i}_count" for i in range(17)]
rel_cols = [f"week_{i}_rel_freq" for i in range(17)]
like_cols = [f"week_{i}_likes" for i in range(17)]
video_cols = [f"week_{i}_n_videos" for i in range(17)]

features["weeks_nonzero"] = (wide[count_cols] > 0).sum(axis=1)

features["count_mean_0_16"] = wide[count_cols].mean(axis=1)
features["count_std_0_16"] = wide[count_cols].std(axis=1)
features["count_max_0_16"] = wide[count_cols].max(axis=1)

features["rel_freq_mean_0_16"] = wide[rel_cols].mean(axis=1)
features["rel_freq_std_0_16"] = wide[rel_cols].std(axis=1)
features["rel_freq_max_0_16"] = wide[rel_cols].max(axis=1)

features["likes_mean_0_16"] = wide[like_cols].mean(axis=1)
features["likes_std_0_16"] = wide[like_cols].std(axis=1)
features["likes_max_0_16"] = wide[like_cols].max(axis=1)

features["n_videos_mean_0_16"] = wide[video_cols].mean(axis=1)
features["n_videos_max_0_16"] = wide[video_cols].max(axis=1)

# recent momentum: weeks 14-16 vs 11-13
features["rel_freq_recent3_mean"] = wide[[f"week_{i}_rel_freq" for i in range(14, 17)]].mean(axis=1)
features["rel_freq_prev3_mean"] = wide[[f"week_{i}_rel_freq" for i in range(11, 14)]].mean(axis=1)

features["rel_freq_momentum_3w"] = (
    features["rel_freq_recent3_mean"] - features["rel_freq_prev3_mean"]
)

features["likes_recent3_mean"] = wide[[f"week_{i}_likes" for i in range(14, 17)]].mean(axis=1)
features["likes_prev3_mean"] = wide[[f"week_{i}_likes" for i in range(11, 14)]].mean(axis=1)

features["likes_momentum_3w"] = (
    features["likes_recent3_mean"] - features["likes_prev3_mean"]
)

# simple linear trend over weeks 0-16
weeks = np.arange(17)

def slope(row):
    return np.polyfit(weeks, row.values, 1)[0]

features["rel_freq_slope_0_16"] = wide[rel_cols].apply(slope, axis=1)
features["count_slope_0_16"] = wide[count_cols].apply(slope, axis=1)
features["likes_slope_0_16"] = wide[like_cols].apply(slope, axis=1)

X = pd.DataFrame(features, index=wide.index)
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("y summary:")
print(y.describe())


X shape: (11156, 236)
y shape: (11156,)
y summary:
count    11156.000000
mean        -2.503701
std          4.892635
min        -16.230565
25%         -1.803382
50%         -0.693792
75%         -0.051088
max         16.050188
dtype: float64


In [14]:
def spearman_score(y_true, y_pred):
    return spearmanr(y_true, y_pred).statistic

def pearson_score(y_true, y_pred):
    return pearsonr(y_true, y_pred)[0]

cv = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "linear": make_pipeline(
        StandardScaler(),
        LinearRegression()
    ),

    "ridge": make_pipeline(
        StandardScaler(),
        Ridge(alpha=1.0)
    ),

    "lasso": make_pipeline(
        StandardScaler(),
        Lasso(alpha=0.001, max_iter=10000)
    ),

    "random_forest": RandomForestRegressor(
        n_estimators=500,
        max_depth=12,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    ),

    "extra_trees": ExtraTreesRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),

    "hist_gradient_boosting": HistGradientBoostingRegressor(
        max_iter=500,
        learning_rate=0.03,
        max_leaf_nodes=31,
        random_state=42
    ),

    "xgboost": XGBRegressor(
        objective="reg:squarederror",
        n_estimators=600,
        max_depth=5,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        reg_alpha=0.0,
        random_state=42,
        n_jobs=-1
    ),

    "mlp_small": make_pipeline(
        StandardScaler(),
        MLPRegressor(
            hidden_layer_sizes=(128, 64),
            activation="relu",
            alpha=1e-4,
            learning_rate_init=1e-3,
            batch_size=128,
            max_iter=500,
            early_stopping=True,
            validation_fraction=0.15,
            random_state=42
        )
    ),

    "mlp_large": make_pipeline(
        StandardScaler(),
        MLPRegressor(
            hidden_layer_sizes=(256, 128, 64),
            activation="relu",
            alpha=1e-4,
            learning_rate_init=5e-4,
            batch_size=128,
            max_iter=600,
            early_stopping=True,
            validation_fraction=0.15,
            random_state=42
        )
    ),
}

scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "medae": make_scorer(
        median_absolute_error,
        greater_is_better=False
    ),
    "r2": "r2",
    "explained_variance": make_scorer(
        explained_variance_score
    ),
    "spearman": make_scorer(spearman_score),
    "pearson": make_scorer(pearson_score),
}

rows = []

for name, model in models.items():

    print(f"Training {name}...")

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=True
    )

    rows.append({
        "model": name,

        "train_rmse_mean": -scores["train_rmse"].mean(),
        "train_rmse_std": scores["train_rmse"].std(),

        "test_rmse_mean": -scores["test_rmse"].mean(),
        "test_rmse_std": scores["test_rmse"].std(),

        "test_mae_mean": -scores["test_mae"].mean(),
        "test_mae_std": scores["test_mae"].std(),

        "test_median_ae_mean": -scores["test_medae"].mean(),
        "test_median_ae_std": scores["test_medae"].std(),

        "test_r2_mean": scores["test_r2"].mean(),
        "test_r2_std": scores["test_r2"].std(),

        "test_explained_variance_mean": scores["test_explained_variance"].mean(),
        "test_explained_variance_std": scores["test_explained_variance"].std(),

        "test_spearman_mean": scores["test_spearman"].mean(),
        "test_spearman_std": scores["test_spearman"].std(),

        "test_pearson_mean": scores["test_pearson"].mean(),
        "test_pearson_std": scores["test_pearson"].std(),
    })

cv_results = (
    pd.DataFrame(rows)
    .sort_values("test_spearman_mean", ascending=False)
    .reset_index(drop=True)
)

cv_results


Training linear...
Training ridge...
Training lasso...
Training random_forest...
Training extra_trees...
Training hist_gradient_boosting...
Training xgboost...
Training mlp_small...
Training mlp_large...


,model,train_rmse_mean,train_rmse_std,test_rmse_mean,test_rmse_std,test_mae_mean,test_mae_std,test_median_ae_mean,test_median_ae_std,test_r2_mean,test_r2_std,test_explained_variance_mean,test_explained_variance_std,test_spearman_mean,test_spearman_std,test_pearson_mean,test_pearson_std
0,random_forest,2.583674,0.013020,3.749517,0.018111,2.314298,0.022887,1.016008,0.038655,0.411898,0.014960,0.412082,0.014837,0.461943,0.018402,0.642237,0.011133
1,xgboost,2.280144,0.012999,3.771786,0.009235,2.327478,0.017468,1.014168,0.024235,0.404790,0.018409,0.405029,0.018160,0.458104,0.016576,0.637361,0.013010
2,hist_gradient_boosting,2.009132,0.010126,3.785746,0.018459,2.323841,0.021648,1.022381,0.023801,0.400398,0.018098,0.400650,0.017891,0.455576,0.017107,0.634722,0.012550
3,extra_trees,0.709016,0.009286,3.795130,0.023216,2.357882,0.019464,1.043958,0.039958,0.397357,0.020513,0.397639,0.020177,0.441299,0.021636,0.631100,0.015240
4,mlp_large,3.458813,0.251877,4.223685,0.050814,2.649446,0.031524,1.140131,0.039955,0.253606,0.026595,0.255434,0.025368,0.330679,0.038457,0.519975,0.021025
5,mlp_small,3.550671,0.233825,4.245571,0.076947,2.769554,0.075546,1.354342,0.100973,0.246445,0.006024,0.248841,0.003481,0.323954,0.020803,0.509090,0.003695
6,lasso,4.268646,0.013263,4.334242,0.058522,2.795393,0.028768,1.371560,0.051317,0.214112,0.026572,0.214565,0.025901,0.311787,0.024620,0.464350,0.025597
7,ridge,4.265182,0.013060,4.342015,0.056545,2.813641,0.025361,1.405028,0.051390,0.211272,0.026953,0.211747,0.026236,0.309019,0.024453,0.461781,0.025549
8,linear,4.265007,0.013035,4.343702,0.055618,2.817490,0.024618,1.414717,0.050055,0.210657,0.026893,0.211139,0.026159,0.308229,0.024457,0.461216,0.025408


In [21]:
# current best model
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

# out-of-fold predictions
y_pred = cross_val_predict(
    model,
    X,
    y,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1
)

eval_df = pd.DataFrame({
    "word": X.index,
    "y_true": y,
    "y_pred": y_pred
}).reset_index(drop=True)

# sanity-check metrics
print("RMSE:", mean_squared_error(eval_df["y_true"], eval_df["y_pred"]) ** 0.5)
print("MAE:", mean_absolute_error(eval_df["y_true"], eval_df["y_pred"]))
print("R2:", r2_score(eval_df["y_true"], eval_df["y_pred"]))
print("Spearman:", spearmanr(eval_df["y_true"], eval_df["y_pred"]).statistic)
print("Pearson:", pearsonr(eval_df["y_true"], eval_df["y_pred"])[0])


RMSE: 3.751800289958898
MAE: 2.315231121636044
R2: 0.411924990852641
Spearman: 0.46228365827144935
Pearson: 0.641891419135053


In [22]:
def ranking_eval(df, k):
    top_pred = df.sort_values("y_pred", ascending=False).head(k)
    top_true = df.sort_values("y_true", ascending=False).head(k)

    top_true_words = set(top_true["word"])
    precision_at_k = top_pred["word"].isin(top_true_words).mean()

    return {
        "k": k,
        "overlap_count": int(top_pred["word"].isin(top_true_words).sum()),
        "precision_at_k": precision_at_k,
        "avg_true_y_pred_topk": top_pred["y_true"].mean(),
        "avg_true_y_ideal_topk": top_true["y_true"].mean(),
        "median_true_y_pred_topk": top_pred["y_true"].median(),
        "median_true_y_ideal_topk": top_true["y_true"].median(),
    }

ranking_results = pd.DataFrame([
    ranking_eval(eval_df, k)
    for k in [10, 20, 50, 100, 200]
])

ranking_results


,k,overlap_count,precision_at_k,avg_true_y_pred_topk,avg_true_y_ideal_topk,median_true_y_pred_topk,median_true_y_ideal_topk
0,10,0,0.00,13.817054,15.474925,13.569743,15.398461
1,20,5,0.25,13.873928,15.031855,13.569743,15.133917
2,50,48,0.96,13.689625,13.801545,13.713584,13.713584
3,100,76,0.76,8.763318,8.983482,5.263695,5.263695
4,200,140,0.70,5.438206,5.849521,3.203906,3.435627


In [23]:
eval_df.sort_values("y_pred", ascending=False).head(50)


,word,y_true,y_pred
8294,reyn,13.492668,14.227001
985,bandicoots,13.310347,14.227001
10534,vanillaware,14.339965,14.227001
3767,ffbe,14.185814,14.227001
5406,karia,13.310347,14.227001
7248,peste,13.310347,14.227001
1514,burlington,13.646818,14.227001
7176,peakuniku,13.492668,14.227001
5561,kurumi,14.099574,14.227001
5213,jack-o,14.981995,14.227001


In [24]:
eval_df.sort_values("y_true", ascending=False).head(50)


,word,y_true,y_pred
354,adef,16.050188,14.047866
5328,jord,15.691304,14.157488
4483,guma,15.675142,14.157488
577,anaya,15.573521,14.157488
5389,kairosoft,15.485097,14.047866
3599,exvius,15.311825,14.111249
3839,fissure,15.285839,14.120263
9796,tauros,15.285839,14.111249
6525,muramasa,15.226174,14.120263
2577,darkviperau,15.164317,14.047866
